# 05 - Stable Diffusion in .NET

Text to image, image to image, inpainting, emphasis and LoRA, on the ONNX export of Stable
Diffusion 1.5. With the same seed and settings the pipeline reproduces diffusers' own ONNX
pipelines: to the pixel on float32 exports, to about 55 dB PSNR on this float16 one.

The first run downloads about 2 GB. A 512x512 image at 25 steps takes a few minutes on a laptop CPU.

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil.*

In [ ]:
#r "nuget: Gravicode.HFNet.GraviHub, 0.7.0"
#r "nuget: Gravicode.HFNet.GraviTokenizers, 0.7.0"
#r "nuget: Gravicode.HFNet.GraviOptimum, 0.7.0"
#r "nuget: Gravicode.HFNet.GraviDiffusers, 0.7.0"
#r "nuget: SixLabors.ImageSharp, 3.1.12"

using Gravicode.HFNet.GraviDiffusers;
using SixLabors.ImageSharp;
using SixLabors.ImageSharp.PixelFormats;

In [ ]:
// The sampler is built from the repository's scheduler_config.json (Sampler.Auto).
using var pipeline = DiffusionPipeline.FromPretrained("nmkd/stable-diffusion-1.5-onnx-fp16");
Console.WriteLine($"{pipeline}, VAE factor {pipeline.VaeScaleFactor}");

## 1. Text to image

In [ ]:
var options = new GenerationOptions(Steps: 25, GuidanceScale: 7.5, Width: 512, Height: 512, Seed: 42,
                                    NegativePrompt: "blurry, low quality");
using var lighthouse = pipeline.Generate("a red lighthouse on a cliff at dawn, oil painting", options,
    (step, total) => { if (step % 5 == 0) Console.WriteLine($"step {step}/{total}"); });
lighthouse.Save("lighthouse.png");

## 2. Image to image

The picture is encoded, noised to `strength` of the way through the schedule and denoised from there.
The VAE encoder's in-graph sampling is replaced by its mean, so a seed reproduces a result - the
export as published returns a different latent on every call, in Python too.

In [ ]:
using var night = pipeline.ImageToImage("a red lighthouse on a cliff at night, stars, oil painting",
                                        lighthouse, strength: 0.6, options);
night.Save("lighthouse-night.png");

## 3. Inpainting

White in the mask is repainted. A 9-channel inpainting UNet is used as diffusers uses it; an ordinary
UNet like this one blends instead, replacing the unmasked area with the original after every step.

In [ ]:
using var mask = new Image<Rgb24>(512, 512);
mask.ProcessPixelRows(rows =>
{
    for (var y = 40; y < 200; y++)
    {
        var row = rows.GetRowSpan(y);
        for (var x = 300; x < 480; x++) row[x] = new Rgb24(255, 255, 255);
    }
});

using var moon = pipeline.Inpaint("a full moon in the sky", lighthouse, mask, options);
moon.Save("lighthouse-moon.png");

## 4. Emphasis and LoRA

`(word)`, `[word]` and `(word:1.4)` weight parts of a prompt, AUTOMATIC1111's syntax and arithmetic.
A LoRA - kohya's layout or diffusers' - is merged into the UNet and text encoder's ONNX weights
without rewriting the files.

In [ ]:
pipeline.PromptWeighting = true;
PromptWeights.Parse("a (red:1.4) lighthouse at [dawn], ((oil painting))")

In [ ]:
// var report = pipeline.LoadLora("some-user/some-sd15-lora", scale: 0.8);
// Console.WriteLine(report);
// pipeline.UnloadLoras();